In [25]:
# Import pandas
import pandas as pd

In [26]:
# Importing data for analysis
df = pd.read_csv("forex_data.csv")

In [27]:
# Copy of original data
df_original = df.copy()

In [28]:
df.head()

,slug,date,open,high,low,close,currency
0,GBP/EGP,10-04-2001,5.58090,5.5947,5.5947,5.5947,EGP
1,GBP/EGP,04-06-2001,5.47517,5.4939,5.4939,5.4939,EGP
2,GBP/EGP,01-08-2001,5.67990,5.6543,5.6543,5.6543,EGP
3,GBP/EGP,29-07-2002,7.21700,7.2170,7.2170,7.2170,EGP
4,GBP/EGP,02-01-2003,7.42429,7.3899,7.3899,7.3899,EGP


In [29]:
# Convert date to datetime
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')

In [30]:
# Sort values
df = df.sort_values(by=['slug', 'date'])

df.head()


,slug,date,open,high,low,close,currency
970347,AUD/BRL,2007-05-29,1.5894,1.5965,1.5836,1.5920,BRL
970348,AUD/BRL,2007-05-30,1.5920,1.6083,1.5902,1.6002,BRL
970349,AUD/BRL,2007-05-31,1.6006,1.6067,1.5864,1.5896,BRL
970350,AUD/BRL,2007-06-01,1.5904,1.5970,1.5751,1.5796,BRL
970351,AUD/BRL,2007-06-04,1.5806,1.5806,1.5806,1.5806,BRL


In [31]:
# Check the duplicate rows
print(df.duplicated().sum()) 
df = df.drop_duplicates()

0


In [32]:
# Count number of records per currency pair
currency_counts = df.groupby('slug').size().reset_index(name='count')
print(currency_counts)

        slug  count
0    AUD/BRL   3552
1    AUD/CAD   4602
2    AUD/CHF   4573
3    AUD/CNY   4584
4    AUD/CZK   4106
..       ...    ...
241  USD/VND   4600
242  USD/XOF   4593
243  USD/XPF   4600
244  USD/ZAR   4596
245  USD/ZMW   2234

[246 rows x 2 columns]


In [33]:
# Calculate returns for all pairs
df['returns'] = df.groupby('slug')['close'].pct_change()

In [34]:
# Calculating volatility for each pair
df['volatility_30'] = df.groupby('slug')['returns'].rolling(30).std().reset_index(level=0, drop=True)

In [35]:
# Buy units at the FIRST closing price of each pair, then track value over time
investment = 100000

df['entry_price'] = df.groupby('slug')['close'].transform('first')
df['units'] = investment / df['entry_price']  
df['value'] = df['units'] * df['close']        
df['pnl'] = df.groupby('slug')['value'].transform(lambda x: x - x.iloc[0])

print(df[['slug', 'date', 'close', 'units', 'value', 'pnl']].head(10))

           slug       date   close         units          value          pnl
970347  AUD/BRL 2007-05-29  1.5920  62814.070352  100000.000000     0.000000
970348  AUD/BRL 2007-05-30  1.6002  62814.070352  100515.075377   515.075377
970349  AUD/BRL 2007-05-31  1.5896  62814.070352   99849.246231  -150.753769
970350  AUD/BRL 2007-06-01  1.5796  62814.070352   99221.105528  -778.894472
970351  AUD/BRL 2007-06-04  1.5806  62814.070352   99283.919598  -716.080402
970352  AUD/BRL 2007-11-28  1.5791  62814.070352   99189.698492  -810.301508
970353  AUD/BRL 2007-11-29  1.5656  62814.070352   98341.708543 -1658.291457
970354  AUD/BRL 2007-11-30  1.5729  62814.070352   98800.251256 -1199.748744
970355  AUD/BRL 2007-12-03  1.5689  62814.070352   98548.994975 -1451.005025
970356  AUD/BRL 2007-12-04  1.5855  62814.070352   99591.708543  -408.291457


In [36]:
# OHLC Features
df['range'] = df['high'] - df['low']
df['price_change'] = df['close'] - df['open']

In [37]:
# Cumulative return (how much has the investment grown over time)
df['cumulative_return'] = df.groupby('slug')['returns'].transform(
    lambda x: (1 + x).cumprod() - 1
)

In [38]:
# Show top performing pairs by final cumulative return
pd.set_option('display.float_format', '{:.4f}'.format)
top_returns = df.groupby('slug')['cumulative_return'].last().sort_values(ascending=False)
print('Top 10 Best Performing Pairs:')
print(top_returns.head(10))

Top 10 Best Performing Pairs:
slug
USD/IQD   4880.3151
USD/MMK    264.9842
EUR/MMK    245.4990
USD/ARS     96.7587
EUR/ARS     28.5984
GBP/ARS     24.1362
GBP/CUP     21.8217
USD/UZS     13.9462
USD/BYN      7.4947
CHF/TRY      7.0678
Name: cumulative_return, dtype: float64


In [39]:
# Show lowest performing pairs by final cumulative return
print('Bottom 10 Worst Performing Pairs:')
print(top_returns.tail(10))

Bottom 10 Worst Performing Pairs:
slug
INR/CAD   -0.3980
GBP/ILS   -0.4166
GBP/CZK   -0.4197
GBP/CHF   -0.4246
INR/NZD   -0.4405
INR/TWD   -0.4739
INR/THB   -0.4923
INR/CHF   -0.5605
JPY/CHF   -0.9930
GBP/TRY   -0.9956
Name: cumulative_return, dtype: float64


In [40]:
# Export raw untouched data
df_original.to_csv('forex_original.csv', index=False)
print("Original data exported!")

Original data exported!


In [41]:
# Exporting the dataset
df.to_csv("forex_processed.csv", index=False)
print("Saved!")

Saved!


In [42]:
# Export grouped summary — count, avg return, avg volatility per pair
summary = df.groupby('slug').agg(
    record_count   = ('close', 'count'),
    avg_close      = ('close', 'mean'),
    avg_volatility = ('volatility_30', 'mean'),
    cumulative_ret = ('cumulative_return', 'last'),
    final_pnl      = ('pnl', 'last')
).round(4).reset_index()

summary.to_csv('forex_summary.csv', index=False)
print("Summary report exported!")
print(summary.head())

Summary report exported!
      slug  record_count  avg_close  avg_volatility  cumulative_ret  \
0  AUD/BRL          3552     2.3294          0.0099          1.3742   
1  AUD/CAD          4602     0.9520          0.0056         -0.0307   
2  AUD/CHF          4573     0.8430          0.0072         -0.2915   
3  AUD/CNY          4584     5.6301          0.0093         -0.2148   
4  AUD/CZK          4106    17.3138          0.0065         -0.1599   

    final_pnl  
0 137418.3417  
1  -3072.0428  
2 -29154.4850  
3 -21475.0578  
4 -15989.3629  


In [43]:
# Verify all exports worked
import os

files = ['forex_original.csv', 'forex_processed.csv', 'forex_summary.csv']
for f in files:
    size = os.path.getsize(f) / (1024*1024)  # size in MB
    print(f"{f} — {size:.2f} MB")

forex_original.csv — 57.14 MB
forex_processed.csv — 218.50 MB
forex_summary.csv — 0.01 MB
